# Algorithm $\epsilon$-greedy decaying for the Problem of Bandits

The Problem of Bandits is specific Dynamic Programming problem, which can be modelled by a MDP with:
- $\mathcal{S}=\mathcal{A}=\{1,2,\dots,K\}$ machines or arms;
- the rewards are i.i.d. of unknown laws.


The $\epsilon$-greedy decaying is an algorithm to select the optimal machine, by maximing the rewards. If follows the same explore-then-commit logic of ETC(m), but introducing a randomness in the choice of one of two phases.


**$\epsilon$-greedy decaying Algorithm**

This algorithm is a $\epsilon$-greedy algorithm with $\epsilon:=(\epsilon_{n})_{n}$ a decreasing sequence. It is carachterised by two phases at each time $n\in\mathbb{N}^{+}$:

- *exploration, with probability $\epsilon_{n}$*: a machine in $\mathcal{S}$ is selected randomly uniformly and played;

- *exploitation, with probability $1-\epsilon_{n}$*: the best action is update by chosing $a^{*}\in argmax_{a} \hat{\mu}_{n-1}(a)$.



**Example**

Consider a model of bandits with $K=3$ machines having i.i.d. Poisson distributed rewards:

$$
\mathbb{P}_1=Poi(2),\quad \mathbb{P}_2=Poi(3), \quad\mathbb{P}_3=Poi(4).
$$

We implement the $\epsilon$-greedy algorithm with the choice of decreasing sequence:

$$
\epsilon_{n} = \min\left\{ 1, \dfrac{3}{n} \right\}.
$$

In [2]:

import numpy as np

# parameters
lambdas = [2, 3, 4]
K = 3
T = 1000
N = 100

rng = np.random.default_rng(2026)    # random generator with seed 2026

optimal_machines = []                # accounts for the optimal machines among 1, ..., K
total_rewards = []                   # accounts for the final total rewards

# 100 essais
# find best machine + total reward, for each essai
for essai in range(N):

    # initialisation
    recompenses = [[] for i in range(K)]     # extracts the rewards for the best machine
    total = 0                                # sums the total reward at each essai

    # Algorithm epsilon_n-greedy
    for n in range(1, T + 1):

        # Compute epsilon_n
        eps_n = min(1, 3/n)

        # Exploration :
        # if number in [0,1] is < eps_n,
        # select a machine randomly uniformly among 0,..., K-1
        if rng.random() < eps_n:
            machine = rng.integers(K)

        # Otherwise, Exploitation:
        # compute the average empirical rewards and find the highest one,
        # find then the optimal machine.
        # If reward not computed yet for machine i, set the average reward to 0
        else:
            moyennes = np.array([
                np.mean(recompenses[i]) if len(recompenses[i]) > 0 else 0
                for i in range(K)
            ])
            machine = np.argmax(moyennes)


        # Reward from the machine selected in exploration or exploitation phase
        recompense = rng.poisson(lambdas[machine])

        recompenses[machine].append(recompense)      # save result for next loop
        total += recompense                          # sum for total final reward


    # Select the best machine over 0, ..., K-1
    moyennes = np.array([
        np.mean(recompenses[i]) if len(recompenses[i]) > 0 else 0
        for i in range(K)
    ])

    best_machine = np.argmax(moyennes)

    optimal_machines.append(best_machine)     # save best machine in 0, 1, ..., K-1
    total_rewards.append(total)               # save total reward


# Statistics
optimal_machines = np.array(optimal_machines)
total_rewards = np.array(total_rewards)

for i in range(K):
    print(f"Machine {i+1} : the best in {np.sum(optimal_machines == i)}% of times")

print("Average total reward: ", np.mean(total_rewards))

print("Average reward for each trial :", np.mean(total_rewards) / T)

print("Standard deviation :", np.std(total_rewards, ddof=1))


Machine 1 : the best in 0% of times
Machine 2 : the best in 6% of times
Machine 3 : the best in 94% of times
Average total reward:  3892.37
Average reward for each trial : 3.89237
Standard deviation : 240.43116383999504


**Remarks**:

This algorithm is sub-linear, specifically logarithmic. Therefore, it performs better than the classic ETC(m). Indeed, here the machine $3$ is selected in the $94\%$ of times, a percentage way higher than with ETC(m).